# Consultoría InmoValor: Modelado Predictivo mediante Redes Neuronales Artificiales

## Clasificación de propiedades por categoría de precio

*Empresa analizada:* **SmartStay Advisors**  
*Curso:* CC3074 - Minería de Datos  
*Tema:* Redes Neuronales Artificiales para clasificación

*Integrantes de la investigación:*
- Vianka Castro - 23201
- Ricardo Godinez - 23247
- Felipe Aguilar - 23195

---

### Propósito de esta sección

Esta parte del informe desarrolla los primeros pasos de la entrega de Redes Neuronales Artificiales. El objetivo es construir modelos capaces de clasificar las propiedades de Airbnb según la categoría de precio definida previamente: **Económica**, **Intermedia** o **Cara**.

Para que la comparación sea válida frente a los modelos trabajados en entregas anteriores, se conserva la misma lógica metodológica utilizada en el notebook de SVM: limpieza base del conjunto de datos, creación de la variable `categoria_precio`, separación 70/30, semilla `random_state=42` y partición estratificada por categoría de precio.

## Preparación del entorno de trabajo

En esta sección se cargan las librerías necesarias para el análisis. Se utilizan herramientas de `pandas` y `numpy` para manipulación de datos, y componentes de `scikit-learn` para preparar variables, construir pipelines y entrenar redes neuronales de clasificación.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.base import clone

import numpy as np
import pandas as pd
try:
    import pyreadr
except ImportError as exc:
    raise ImportError("No se encontró la librería pyreadr. Instálala con: pip install pyreadr") from exc
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
TEST_SIZE = 0.30


def columnas_categoricas(dataframe, excluir=None):
    """Devuelve columnas categóricas o textuales, excluyendo las que se indiquen."""
    excluir = set(excluir or [])
    cols = []
    for col in dataframe.columns:
        if col in excluir:
            continue
        dtype = dataframe[col].dtype
        if (
            pd.api.types.is_object_dtype(dtype)
            or pd.api.types.is_string_dtype(dtype)
            or isinstance(dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(dtype)
        ):
            cols.append(col)
    return cols


def crear_onehot_encoder():
    """Crea un OneHotEncoder compatible con versiones nuevas y anteriores de scikit-learn."""
    try:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=True)
    except TypeError:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse=True)

print('Entorno preparado correctamente.')

Entorno preparado correctamente.


## Carga del conjunto de datos

El conjunto de datos utilizado es `listings.RData`, el mismo archivo empleado en las entregas anteriores. La variable `price` se convierte a formato numérico porque usualmente viene como texto con símbolos de moneda o separadores de miles.

In [2]:
ruta_datos = Path('listings.RData')

if not ruta_datos.exists():
    raise FileNotFoundError(
        "No se encontró el archivo 'listings.RData'. "
        "Coloca este notebook en la misma carpeta del archivo de datos o actualiza la variable ruta_datos."
    )

resultado = pyreadr.read_r(str(ruta_datos))
df = list(resultado.values())[0]

print(f'Dimensiones iniciales del dataset: {df.shape}')
print('Primeras columnas disponibles:')
print(df.columns[:12].tolist())

# Conversión robusta de price a numérico.
precio_limpio = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['price'] = pd.to_numeric(precio_limpio, errors='coerce')

print('Conversión de price completada.')
print(df['price'].describe().round(2))

Dimensiones iniciales del dataset: (171748, 80)
Primeras columnas disponibles:
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name']
Conversión de price completada.
count    76246.00
mean       750.51
std       4250.61
min          8.00
25%        120.00
50%        193.00
75%        326.00
max      50123.00
Name: price, dtype: float64


## Limpieza inicial del conjunto de datos

Antes de construir los modelos, se eliminan registros sin precio, se descartan columnas completamente vacías y se imputan valores faltantes. Este paso permite que el preprocesamiento posterior trabaje con datos consistentes sin perder observaciones innecesariamente.

In [3]:
df = df.replace([np.inf, -np.inf], np.nan)

faltantes = df.isnull().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)

print('=== CANTIDAD DE COLUMNAS CON DATOS VACÍOS ===')
print(f'Columnas con vacíos: {len(faltantes)}')
print('Top 10 columnas con más vacíos:')
print(faltantes.head(10))

print('\n=== PORCENTAJE DE VACÍOS TOP 10 ===')
porcentajes = (faltantes / len(df)) * 100
print((porcentajes.head(10).round(2)).astype(str) + ' %')

# Se elimina únicamente la fila sin variable objetivo de precio.
df = df.dropna(subset=['price']).copy()
df = df.dropna(axis=1, how='all')

numericas_df = df.select_dtypes(include=['number']).columns.tolist()
if 'price' in numericas_df:
    numericas_df.remove('price')

categoricas_df = columnas_categoricas(df, excluir=['price'])

if numericas_df:
    imputador_num = SimpleImputer(strategy='median')
    df[numericas_df] = imputador_num.fit_transform(df[numericas_df])

if categoricas_df:
    imputador_cat = SimpleImputer(strategy='constant', fill_value='Sin Dato')
    df[categoricas_df] = imputador_cat.fit_transform(df[categoricas_df])

print('\nFilas sin precio eliminadas y faltantes imputados.')
print('Total de nulos restantes:', int(df.isnull().sum().sum()))
print('Dimensiones luego de limpieza:', df.shape)

=== CANTIDAD DE COLUMNAS CON DATOS VACÍOS ===
Columnas con vacíos: 23
Top 10 columnas con más vacíos:
calendar_updated                171748
price                            95502
estimated_revenue_l365d          95502
neighbourhood_group_cleansed     50683
review_scores_value              40328
review_scores_location           40328
review_scores_checkin            40324
review_scores_accuracy           40312
review_scores_communication      40308
review_scores_cleanliness        40302
dtype: int64

=== PORCENTAJE DE VACÍOS TOP 10 ===
calendar_updated                100.0 %
price                           55.61 %
estimated_revenue_l365d         55.61 %
neighbourhood_group_cleansed    29.51 %
review_scores_value             23.48 %
review_scores_location          23.48 %
review_scores_checkin           23.48 %
review_scores_accuracy          23.47 %
review_scores_communication     23.47 %
review_scores_cleanliness       23.47 %
dtype: str

Filas sin precio eliminadas y faltantes imputa

## Variable respuesta: categoría de precio

Para clasificar las propiedades, el precio se transforma en una variable categórica de tres niveles usando terciles. Esta decisión mantiene clases balanceadas y permite que la red neuronal aprenda a diferenciar propiedades **Económicas**, **Intermedias** y **Caras**.

Las columnas binarias `es_economica`, `es_intermedia` y `es_cara` se conservan como apoyo descriptivo, aunque el modelo multiclase utilizará directamente `categoria_precio`.

In [4]:
q1, q2 = df['price'].astype(float).quantile([1/3, 2/3])

df['categoria_precio'] = pd.cut(
    df['price'].astype(float),
    bins=[-float('inf'), q1, q2, float('inf')],
    labels=['Economica', 'Intermedia', 'Cara'],
    include_lowest=True
)

df['es_economica'] = (df['categoria_precio'] == 'Economica').astype(int)
df['es_intermedia'] = (df['categoria_precio'] == 'Intermedia').astype(int)
df['es_cara'] = (df['categoria_precio'] == 'Cara').astype(int)

print('Puntos de corte usados para categoria_precio:')
print(f'q1 tercil bajo: {q1:.2f}')
print(f'q2 tercil alto: {q2:.2f}')

print('\nDistribución de categoria_precio:')
distribucion_categoria = df['categoria_precio'].value_counts().sort_index()
print(distribucion_categoria)
print('\nProporción por categoría:')
print((distribucion_categoria / len(df)).round(4))

print('\nEjemplo de codificación:')
display(df[['price', 'categoria_precio', 'es_economica', 'es_intermedia', 'es_cara']].head())

Puntos de corte usados para categoria_precio:
q1 tercil bajo: 143.00
q2 tercil alto: 268.00

Distribución de categoria_precio:
categoria_precio
Economica     25689
Intermedia    25153
Cara          25404
Name: count, dtype: int64

Proporción por categoría:
categoria_precio
Economica     0.3369
Intermedia    0.3299
Cara          0.3332
Name: count, dtype: float64

Ejemplo de codificación:


,price,categoria_precio,es_economica,es_intermedia,es_cara
0,97.0,Economica,1,0,0
1,160.0,Intermedia,0,1,0
2,38.0,Economica,1,0,0
3,145.0,Intermedia,0,1,0
4,58.0,Economica,1,0,0


## Conformación de variables predictoras

Se retiran columnas que no deben usarse como predictores directos del modelo, especialmente identificadores, fechas de extracción, coordenadas y la propia variable objetivo. También se limita el uso de variables categóricas a aquellas con una cardinalidad manejable para evitar una matriz excesivamente grande después del `OneHotEncoder`.

In [5]:
filas_sanas = df['categoria_precio'].notna()
df_limpio = df[filas_sanas].copy()

columnas_a_botar = [
    'price', 'categoria_precio', 'es_economica', 'es_intermedia', 'es_cara',
    'id', 'scrape_id', 'host_id', 'last_scraped', 'calendar_last_scraped',
    'first_review', 'last_review',
    'latitude', 'longitude',
    'neighbourhood_group_cleansed',
    'host_verifications'
]

X = df_limpio.drop(columns=[col for col in columnas_a_botar if col in df_limpio.columns])
X = X.replace([np.inf, -np.inf], np.nan)

# Se eliminan columnas auxiliares binarias mal codificadas que pueden duplicar información.
columnas_falsas = [col for col in X.columns if col.endswith('_f') and X[col].nunique(dropna=True) <= 2]
X = X.drop(columns=columnas_falsas, errors='ignore')

# Algunas variables pueden venir como texto aunque conceptualmente sean numéricas.
columnas_forzar_numero = ['accommodates', 'bathrooms', 'bedrooms', 'beds']
for col in columnas_forzar_numero:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')

numericas = X.select_dtypes(include=['number']).columns.tolist()
categoricas_crudas = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
categoricas = [col for col in categoricas_crudas if X[col].nunique(dropna=True) < 50]

if categoricas:
    X[categoricas] = X[categoricas].astype(str)

Y_strata = df_limpio['categoria_precio']
Y_dicotomicas = df_limpio[['es_economica', 'es_intermedia', 'es_cara']]

print('Variables predictoras preparadas:')
print(f'- Total de columnas X: {X.shape[1]}')
print(f'- Variables numéricas usadas: {len(numericas)}')
print(f'- Variables categóricas usadas para OneHotEncoder: {len(categoricas)}')
print(f'- Columnas categóricas excluidas por alta cardinalidad: {len(categoricas_crudas) - len(categoricas)}')

Variables predictoras preparadas:
- Total de columnas X: 67
- Variables numéricas usadas: 30
- Variables categóricas usadas para OneHotEncoder: 10
- Columnas categóricas excluidas por alta cardinalidad: 27


## Mismos conjuntos de entrenamiento y prueba

Para conservar comparabilidad con los modelos anteriores, se usa una partición de **70% entrenamiento** y **30% prueba**, con `random_state=42`. Además, se aplica `stratify=Y_strata` para que las tres categorías de precio mantengan proporciones similares en entrenamiento y prueba.

In [6]:
X_train_clf, X_test_clf, Y_train_dico, Y_test_dico = train_test_split(
    X,
    Y_dicotomicas,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=Y_strata
)

# scikit-learn puede generar error si la RNA recibe etiquetas de texto
# y tiene early_stopping=True. Por eso se codifica la variable respuesta
# para entrenar, pero se conservan etiquetas legibles para el informe.
label_encoder_rna = LabelEncoder()
label_encoder_rna.fit(Y_strata.astype(str))

y_train_rna = label_encoder_rna.transform(Y_strata.loc[X_train_clf.index].astype(str))
y_test_rna = label_encoder_rna.transform(Y_strata.loc[X_test_clf.index].astype(str))

y_train_rna_etiquetas = pd.Series(
    label_encoder_rna.inverse_transform(y_train_rna),
    index=X_train_clf.index,
    name='categoria_precio'
)
y_test_rna_etiquetas = pd.Series(
    label_encoder_rna.inverse_transform(y_test_rna),
    index=X_test_clf.index,
    name='categoria_precio'
)

preprocesador_clf = ColumnTransformer(
    transformers=[
        ('numeros', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numericas),
        ('textos', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', crear_onehot_encoder())
        ]), categoricas)
    ]
)

print('Resumen del split aplicado:')
print(f'test_size={TEST_SIZE} | random_state={RANDOM_STATE} | stratify=Y_strata')
print(f'X_train_clf: {X_train_clf.shape} | X_test_clf: {X_test_clf.shape}')

print('\nCodificación de la variable respuesta para RNA:')
for codigo, etiqueta in enumerate(label_encoder_rna.classes_):
    print(f'{codigo} = {etiqueta}')

print('\nDistribución relativa en entrenamiento:')
print(y_train_rna_etiquetas.value_counts(normalize=True).sort_index().round(4))

print('\nDistribución relativa en prueba:')
print(y_test_rna_etiquetas.value_counts(normalize=True).sort_index().round(4))

Resumen del split aplicado:
test_size=0.3 | random_state=42 | stratify=Y_strata
X_train_clf: (53372, 67) | X_test_clf: (22874, 67)

Codificación de la variable respuesta para RNA:
0 = Cara
1 = Economica
2 = Intermedia

Distribución relativa en entrenamiento:
categoria_precio
Cara          0.3332
Economica     0.3369
Intermedia    0.3299
Name: proportion, dtype: float64

Distribución relativa en prueba:
categoria_precio
Cara          0.3332
Economica     0.3369
Intermedia    0.3299
Name: proportion, dtype: float64


## Diseño de los modelos de Redes Neuronales Artificiales

Se construyen dos modelos de clasificación con arquitecturas distintas para cumplir con el análisis comparativo:

| Modelo | Topología | Activación | Justificación |
|---|---:|---|---|
| RNA 1 | `(64, 32)` | `relu` | Modelo moderado, eficiente y adecuado para relaciones no lineales sin excesiva complejidad. |
| RNA 2 | `(128, 64, 32)` | `tanh` | Modelo más profundo, con activación diferente para evaluar si una representación más flexible mejora la clasificación. |

Ambos modelos usan el mismo preprocesamiento y el mismo conjunto de entrenamiento y prueba. Esto evita que la comparación se vea afectada por diferencias en preparación de datos.

In [7]:
rna_modelo_1 = Pipeline(steps=[
    ('preprocesador', clone(preprocesador_clf)),
    ('clasificador', MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=0.0001,
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=RANDOM_STATE
    ))
])

rna_modelo_2 = Pipeline(steps=[
    ('preprocesador', clone(preprocesador_clf)),
    ('clasificador', MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation='tanh',
        solver='adam',
        alpha=0.0005,
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=RANDOM_STATE
    ))
])

print('Modelos definidos correctamente.')
print('RNA 1: hidden_layer_sizes=(64, 32), activation=relu')
print('RNA 2: hidden_layer_sizes=(128, 64, 32), activation=tanh')

Modelos definidos correctamente.
RNA 1: hidden_layer_sizes=(64, 32), activation=relu
RNA 2: hidden_layer_sizes=(128, 64, 32), activation=tanh


## Entrenamiento de las redes neuronales

Los dos modelos se entrenan con el mismo conjunto de entrenamiento. El parámetro `early_stopping=True` permite detener el entrenamiento si el desempeño de validación deja de mejorar, lo cual ayuda a evitar entrenamientos innecesariamente largos.

In [8]:
import time

print('Entrenando RNA 1...')
inicio_1 = time.time()
rna_modelo_1.fit(X_train_clf, y_train_rna)
tiempo_rna_1 = time.time() - inicio_1
print(f'RNA 1 entrenada correctamente en {tiempo_rna_1:.2f} segundos.')

print('\nEntrenando RNA 2...')
inicio_2 = time.time()
rna_modelo_2.fit(X_train_clf, y_train_rna)
tiempo_rna_2 = time.time() - inicio_2
print(f'RNA 2 entrenada correctamente en {tiempo_rna_2:.2f} segundos.')

Entrenando RNA 1...
RNA 1 entrenada correctamente en 20.16 segundos.

Entrenando RNA 2...
RNA 2 entrenada correctamente en 74.26 segundos.


## Predicción de la categoría de precio

Una vez entrenados los modelos, se utilizan para predecir la categoría de precio sobre el conjunto de prueba. En este punto ya se cumple el objetivo de usar las redes neuronales para estimar la variable respuesta categórica definida previamente.

In [9]:
y_pred_rna_1_cod = rna_modelo_1.predict(X_test_clf)
y_pred_rna_2_cod = rna_modelo_2.predict(X_test_clf)

# Se regresan las predicciones a texto para que el informe sea legible.
y_pred_rna_1 = label_encoder_rna.inverse_transform(y_pred_rna_1_cod.astype(int))
y_pred_rna_2 = label_encoder_rna.inverse_transform(y_pred_rna_2_cod.astype(int))

predicciones_rna = pd.DataFrame({
    'categoria_real': y_test_rna_etiquetas.values,
    'prediccion_rna_1_relu': y_pred_rna_1,
    'prediccion_rna_2_tanh': y_pred_rna_2
}, index=X_test_clf.index)

print('Predicciones generadas correctamente para ambos modelos.')
print('\nPrimeras predicciones:')
display(predicciones_rna.head(10))

print('\nDistribución de predicciones - RNA 1:')
print(pd.Series(y_pred_rna_1).value_counts().sort_index())

print('\nDistribución de predicciones - RNA 2:')
print(pd.Series(y_pred_rna_2).value_counts().sort_index())

Predicciones generadas correctamente para ambos modelos.

Primeras predicciones:


,categoria_real,prediccion_rna_1_relu,prediccion_rna_2_tanh
22753,Economica,Economica,Economica
5683,Cara,Cara,Cara
161508,Cara,Economica,Economica
9405,Economica,Economica,Economica
6196,Intermedia,Intermedia,Intermedia
28186,Cara,Intermedia,Cara
147833,Economica,Economica,Intermedia
12301,Cara,Cara,Cara
7410,Cara,Cara,Cara
20789,Intermedia,Cara,Cara



Distribución de predicciones - RNA 1:
Cara          7105
Economica     7733
Intermedia    8036
Name: count, dtype: int64

Distribución de predicciones - RNA 2:
Cara          7527
Economica     7955
Intermedia    7392
Name: count, dtype: int64


## Revisión rápida de desempeño preliminar

Aunque las matrices de confusión pertenecen al siguiente punto de la guía, se deja una revisión breve de exactitud y F1 ponderado para confirmar que las predicciones se generaron de forma coherente. La evaluación formal puede continuar en la siguiente sección con las matrices de confusión y el análisis de errores.

In [10]:
resumen_preliminar = pd.DataFrame([
    {
        'Modelo': 'RNA 1 - ReLU',
        'Topología': '(64, 32)',
        'Activación': 'relu',
        'Accuracy prueba': accuracy_score(y_test_rna_etiquetas, y_pred_rna_1),
        'F1 ponderado prueba': f1_score(y_test_rna_etiquetas, y_pred_rna_1, average='weighted'),
        'Tiempo entrenamiento seg.': tiempo_rna_1
    },
    {
        'Modelo': 'RNA 2 - Tanh',
        'Topología': '(128, 64, 32)',
        'Activación': 'tanh',
        'Accuracy prueba': accuracy_score(y_test_rna_etiquetas, y_pred_rna_2),
        'F1 ponderado prueba': f1_score(y_test_rna_etiquetas, y_pred_rna_2, average='weighted'),
        'Tiempo entrenamiento seg.': tiempo_rna_2
    }
])

resumen_preliminar[['Accuracy prueba', 'F1 ponderado prueba', 'Tiempo entrenamiento seg.']] = resumen_preliminar[
    ['Accuracy prueba', 'F1 ponderado prueba', 'Tiempo entrenamiento seg.']
].round(4)

display(resumen_preliminar)

,Modelo,Topología,Activación,Accuracy prueba,F1 ponderado prueba,Tiempo entrenamiento seg.
0,RNA 1 - ReLU,"(64, 32)",relu,0.7098,0.7114,20.1551
1,RNA 2 - Tanh,"(128, 64, 32)",tanh,0.7143,0.7137,74.2594


## Análisis de Resultados

En esta sección se comparan los resultados de ambos modelos de redes neuronales. Se analizan las métricas de desempeño, como exactitud y F1 ponderado, para determinar cuál modelo clasifica mejor las propiedades según la categoría de precio. Se discuten las posibles razones detrás de las diferencias en desempeño, considerando la arquitectura de cada modelo y la naturaleza de los datos. Además, se reflexiona sobre las implicaciones prácticas de los resultados para SmartStay Advisors y se sugieren posibles mejoras o ajustes para futuros modelos.

Se puede notar que los modelos aplicados no alcanzan una exactitud alta, lo que sugiere que la clasificación de propiedades por categoría de precio es un problema complejo que podría beneficiarse de técnicas adicionales, como la ingeniería de características o el uso de modelos más avanzados. Sin embargo, el análisis preliminar proporciona una base sólida para entender cómo las redes neuronales pueden abordar este tipo de problemas y qué aspectos podrían mejorarse en futuras iteraciones.